# LeetCode 2225 — Find Players With Zero or One Losses bằng Python

Link đề: https://leetcode.com/problems/find-players-with-zero-or-one-losses/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Sorting, Counting

Skeleton LeetCode:

```python
class Solution:
    def findWinners(self, matches: List[List[int]]) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa dữ liệu trận đấu thành **bảng đếm số trận thua** của mỗi người chơi, và nhớ rằng người **chỉ thắng** cũng phải xuất hiện trong bảng (với 0 trận thua).
2. Dùng `dict`/`Counter`/`defaultdict` và hàm `dict.setdefault` để vừa "đăng ký" người chơi vừa đếm.
3. Tận dụng giới hạn **id ≤ 10⁵** để dùng **mảng đếm** (đếm phân phối) — có sẵn thứ tự tăng, không cần sắp xếp.
4. Hiểu khác biệt giữa "chưa chơi trận nào" và "chơi nhưng không thua" (giá trị lính canh `−1`).
5. Kiểm thử đối chiếu brute force và kiểm định dạng đầu ra (hai danh sách **tăng dần**).

## 2. Đề bài & ví dụ

`matches[i] = [winner_i, loser_i]`: người `winner_i` thắng người `loser_i`. Trả về `answer` gồm hai danh sách:
- `answer[0]`: mọi người chơi **không thua** trận nào;
- `answer[1]`: mọi người chơi thua **đúng một** trận.

Cả hai danh sách **tăng dần**. Chỉ xét người đã chơi ít nhất một trận. Không có hai trận cùng kết quả.

**Ví dụ 1:** `[[1,3],[2,3],[3,6],[5,6],[5,7],[4,5],[4,8],[4,9],[10,4],[10,9]]` → `[[1,2,10],[4,5,7,8]]`.

**Ví dụ 2:** `[[2,3],[1,3],[5,4],[6,4]]` → `[[1,2,5,6],[]]`.

**Ràng buộc:** `1 ≤ len(matches) ≤ 10⁵`, `1 ≤ winner, loser ≤ 10⁵`, `winner ≠ loser`.

In [1]:
import random
import time
from collections import Counter, defaultdict
from itertools import groupby
from typing import List

examples = [
    ([[1, 3], [2, 3], [3, 6], [5, 6], [5, 7], [4, 5], [4, 8], [4, 9], [10, 4], [10, 9]], [[1, 2, 10], [4, 5, 7, 8]]),
    ([[2, 3], [1, 3], [5, 4], [6, 4]], [[1, 2, 5, 6], []]),
]
for m, exp in examples:
    print(m, "->", exp)

[[1, 3], [2, 3], [3, 6], [5, 6], [5, 7], [4, 5], [4, 8], [4, 9], [10, 4], [10, 9]] -> [[1, 2, 10], [4, 5, 7, 8]]
[[2, 3], [1, 3], [5, 4], [6, 4]] -> [[1, 2, 5, 6], []]


## 3. Kiến thức nền

### 3.1 `Counter` đếm số lần xuất hiện

`Counter(loser for _, loser in matches)` cho số trận thua của mỗi người **đã từng thua**. Người chỉ thắng **không có mặt** trong `Counter` này — phải thêm riêng.

In [2]:
m = examples[1][0]
losses = Counter(l for _, l in m)
print(losses, "| người 1 có trong Counter?", 1 in losses, "| losses[1] =", losses[1])

Counter({3: 2, 4: 2}) | người 1 có trong Counter? False | losses[1] = 0


### 3.2 `dict.setdefault(key, default)`

Trả về `d[key]` nếu có, ngược lại gán `d[key] = default` rồi trả về. Dùng để "đăng ký" người thắng với 0 trận thua **mà không ghi đè** nếu đã có.

In [3]:
d = {3: 2}
d.setdefault(1, 0); d.setdefault(3, 0)
print(d)

{3: 2, 1: 0}


### 3.3 Mảng đếm với lính canh `−1`

Vì id ≤ `10⁵`, dùng `lost = [−1] * (10⁵ + 1)`: `−1` = chưa chơi; khi một người xuất hiện, nâng lên `0` nếu đang `−1`; mỗi trận thua `+1`. Duyệt id tăng dần cho thứ tự sẵn.

In [4]:
lost = [-1] * 8
for w, l in [[2, 3], [1, 3], [5, 4], [6, 4]]:
    if lost[w] == -1: lost[w] = 0
    lost[l] = max(lost[l], 0) + 1
print(list(enumerate(lost)))

[(0, -1), (1, 0), (2, 0), (3, 2), (4, 2), (5, 0), (6, 0), (7, -1)]


### 3.4 `sorted` và list comprehension có điều kiện

`sorted(p for p, c in d.items() if c == 0)` — lọc và sắp trong một dòng.

In [5]:
d = {3: 2, 1: 0, 2: 0, 4: 1}
print(sorted(p for p, c in d.items() if c == 0), sorted(p for p, c in d.items() if c == 1))

[1, 2] [4]


### 3.5 `defaultdict(int)` và `dict.get(k, 0)`

`defaultdict(int)` tự tạo khóa với giá trị `0` khi **đọc** khóa chưa có — tiện để `+= 1`, nhưng chỉ cần **đọc** `d[p]` là `p` đã được thêm vào (có thể làm "bẩn" bảng). `dict.get(k, 0)` đọc mà **không** thêm khóa.

In [6]:
dd = defaultdict(int)
_ = dd[42]                           # chỉ đọc mà khóa 42 đã xuất hiện!
print(dict(dd), "| dict.get không thêm khóa:", {}.get(42, 0), {})

{42: 0} | dict.get không thêm khóa: 0 {}


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1 — lập bảng số trận thua của **mọi người đã chơi**:

| Người | Thắng | Thua | Số trận thua |
|---|---|---|---|
| 1 | 3 | — | 0 |
| 2 | 3 | — | 0 |
| 3 | 6 | 1, 2 | 2 |
| 4 | 5, 8, 9 | 10 | 1 |
| 5 | 6, 7 | 4 | 1 |
| 6 | — | 3, 5 | 2 |
| 7 | — | 5 | 1 |
| 8 | — | 4 | 1 |
| 9 | — | 4, 10 | 2 |
| 10 | 4, 9 | — | 0 |

`answer[0] = [1, 2, 10]`, `answer[1] = [4, 5, 7, 8]` ✓. Chỉ cần **một** con số cho mỗi người: số trận thua. Số trận thắng không liên quan (trừ việc giúp ta biết người đó "đã chơi").

In [7]:
m = examples[0][0]
players = sorted({p for match in m for p in match})
for p in players:
    print(f"người {p:2d}: thắng {[l for w, l in m if w == p]}, thua trước {[w for w, l in m if l == p]}")

người  1: thắng [3], thua trước []
người  2: thắng [3], thua trước []
người  3: thắng [6], thua trước [1, 2]
người  4: thắng [5, 8, 9], thua trước [10]
người  5: thắng [6, 7], thua trước [4]
người  6: thắng [], thua trước [3, 5]
người  7: thắng [], thua trước [5]
người  8: thắng [], thua trước [4]
người  9: thắng [], thua trước [4, 10]
người 10: thắng [4, 9], thua trước []


**Góc nhìn đồ thị.** Coi mỗi người là một đỉnh, mỗi trận `(w, l)` là một cạnh có hướng `w → l`. Khi đó **số trận thua = bậc vào** (in-degree) của đỉnh. `answer[0]` là các đỉnh có bậc vào 0 (nhưng phải có ít nhất một cạnh — tức bậc ra > 0), `answer[1]` là các đỉnh có bậc vào đúng 1. Đây là cùng một phép đếm, nhưng gợi ý cách giải nhiều bài đồ thị khác (Town Judge, sắp xếp tô-pô bắt đầu từ các đỉnh bậc vào 0).

In [8]:
m = examples[0][0]
indeg = Counter(l for _, l in m); outdeg = Counter(w for w, _ in m)
for p in sorted(set(indeg) | set(outdeg)):
    print(f"đỉnh {p:2d}: bậc vào (thua) = {indeg[p]}, bậc ra (thắng) = {outdeg[p]}")

đỉnh  1: bậc vào (thua) = 0, bậc ra (thắng) = 1
đỉnh  2: bậc vào (thua) = 0, bậc ra (thắng) = 1
đỉnh  3: bậc vào (thua) = 2, bậc ra (thắng) = 1
đỉnh  4: bậc vào (thua) = 1, bậc ra (thắng) = 3
đỉnh  5: bậc vào (thua) = 1, bậc ra (thắng) = 2
đỉnh  6: bậc vào (thua) = 2, bậc ra (thắng) = 0
đỉnh  7: bậc vào (thua) = 1, bậc ra (thắng) = 0
đỉnh  8: bậc vào (thua) = 1, bậc ra (thắng) = 0
đỉnh  9: bậc vào (thua) = 2, bậc ra (thắng) = 0
đỉnh 10: bậc vào (thua) = 0, bậc ra (thắng) = 2


## 5. Cách 1 — Brute Force / baseline: đếm lại cho từng người

**Ý tưởng:** lấy tập người chơi; với mỗi người, duyệt toàn bộ `matches` để đếm số trận thua.

In [9]:
def winners_brute(matches):
    players = sorted({p for match in matches for p in match})
    zero, one = [], []
    for p in players:
        cnt = sum(1 for _, l in matches if l == p)
        if cnt == 0: zero.append(p)
        elif cnt == 1: one.append(p)
    return [zero, one]

for m, exp in examples:
    got = winners_brute(m); print(got); assert got == exp

[[1, 2, 10], [4, 5, 7, 8]]
[[1, 2, 5, 6], []]


**Độ phức tạp:** `P` người × `M` trận ⇒ `O(P·M)`; với `M = 10⁵` và `P` tới `2·10⁵` là `2·10¹⁰` — TLE. Space `O(P)`.

**Nút thắt:** mỗi trận chỉ ảnh hưởng tới **một** bộ đếm (của người thua) — có thể đếm tất cả trong **một lượt** qua `matches`.

In [10]:
rng = random.Random(0)
for M in (500, 1000, 2000):
    mm = [[rng.randint(1, M), rng.randint(1, M)] for _ in range(M)]
    mm = [x for x in mm if x[0] != x[1]]
    t = time.perf_counter(); winners_brute(mm)
    print(f"M={M}: {time.perf_counter() - t:.3f}s")

M=500: 0.004s
M=1000: 0.016s
M=2000: 0.067s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** một lượt qua `matches` để xây bảng `lost[p]` = số trận thua của `p`, **đảm bảo mọi người đã chơi đều có mặt** (người thắng được đăng ký với 0 nếu chưa có).

**Quy tắc "nếu… thì…":**
- Với trận `(w, l)`: **nếu** `w` chưa có trong bảng **thì** đặt `lost[w] = 0`; luôn `lost[l] += 1` (đăng ký `l` nếu chưa có).
- Cuối cùng: **nếu** `lost[p] == 0` ⇒ `answer[0]`; **nếu** `lost[p] == 1` ⇒ `answer[1]`.

**Invariant:** sau khi xử lý `t` trận đầu, `lost` chứa đúng những người xuất hiện trong `t` trận đó, mỗi người với số trận thua trong `t` trận.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `dict` + `setdefault` + sắp xếp (bản nộp chính)

In [11]:
def winners_dict(matches):
    lost = {}
    for w, l in matches:
        lost.setdefault(w, 0)
        lost[l] = lost.get(l, 0) + 1
    return [sorted(p for p, c in lost.items() if c == 0),
            sorted(p for p, c in lost.items() if c == 1)]

for m, exp in examples:
    assert winners_dict(m) == exp
print([winners_dict(m) for m, _ in examples])

[[[1, 2, 10], [4, 5, 7, 8]], [[1, 2, 5, 6], []]]


**Độ phức tạp Cách 2:** đếm `O(M)`, sắp `O(P log P)` ⇒ `O(M + P log P)`. Space `O(P)`.

### 7.2 Cách 3 — `Counter` cho thua + `set` cho mọi người chơi

In [12]:
def winners_counter(matches):
    losses = Counter(l for _, l in matches)
    players = {p for match in matches for p in match}
    zero = sorted(p for p in players if losses[p] == 0)
    one = sorted(p for p, c in losses.items() if c == 1)
    return [zero, one]

for m, exp in examples:
    assert winners_counter(m) == exp
print([winners_counter(m) for m, _ in examples])

[[[1, 2, 10], [4, 5, 7, 8]], [[1, 2, 5, 6], []]]


**Độ phức tạp Cách 3:** như Cách 2. Lưu ý `losses[p]` với `Counter` trả 0 cho người chưa thua (không ném lỗi).

### 7.3 Cách 4 — Mảng đếm theo id (không cần sắp xếp)

In [13]:
def winners_array(matches):
    MAXV = 10**5
    lost = [-1] * (MAXV + 1)               # -1: chưa chơi
    for w, l in matches:
        if lost[w] == -1: lost[w] = 0
        if lost[l] == -1: lost[l] = 0
        lost[l] += 1
    zero = [p for p in range(1, MAXV + 1) if lost[p] == 0]
    one = [p for p in range(1, MAXV + 1) if lost[p] == 1]
    return [zero, one]

for m, exp in examples:
    assert winners_array(m) == exp
print([winners_array(m) for m, _ in examples])

[[[1, 2, 10], [4, 5, 7, 8]], [[1, 2, 5, 6], []]]


In [14]:
class SolutionArray:
    def findWinners(self, matches: List[List[int]]) -> List[List[int]]:
        top = max(max(w, l) for w, l in matches)
        lost = [-1] * (top + 1)
        for w, l in matches:
            if lost[w] < 0: lost[w] = 0
            lost[l] = max(lost[l], 0) + 1
        return [[p for p in range(top + 1) if lost[p] == 0],
                [p for p in range(top + 1) if lost[p] == 1]]

print([SolutionArray().findWinners(m) for m, _ in examples])

[[[1, 2, 10], [4, 5, 7, 8]], [[1, 2, 5, 6], []]]


**Độ phức tạp Cách 4:** `O(M + V)` với `V` = id lớn nhất; không có `log`. Space `O(V)`.

### 7.4 Cách 5 — Sắp các người thua rồi `groupby`

Không dùng bảng băm: sắp danh sách người thua, `groupby` để đếm số lần mỗi người thua; người thắng chưa từng thua lấy từ tập người thắng trừ tập người thua.

In [15]:
def winners_sort_group(matches):
    losers = sorted(l for _, l in matches)
    loss_count = {k: len(list(g)) for k, g in groupby(losers)}
    winners = sorted({w for w, _ in matches} - set(loss_count))
    one = [p for p, c in loss_count.items() if c == 1]      # loss_count duyệt theo thứ tự tăng
    return [winners, one]

for m, exp in examples:
    assert winners_sort_group(m) == exp
print([winners_sort_group(m) for m, _ in examples])

[[[1, 2, 10], [4, 5, 7, 8]], [[1, 2, 5, 6], []]]


**Độ phức tạp Cách 5:** `O(M log M)`.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 2 `[[2,3],[1,3],[5,4],[6,4]]`:

| Trận | Thao tác | lost |
|---|---|---|
| (2,3) | đăng ký 2 → 0; 3 thua → 1 | {2:0, 3:1} |
| (1,3) | đăng ký 1 → 0; 3 thua → 2 | {2:0, 3:2, 1:0} |
| (5,4) | đăng ký 5 → 0; 4 thua → 1 | {…, 5:0, 4:1} |
| (6,4) | đăng ký 6 → 0; 4 thua → 2 | {…, 6:0, 4:2} |

`answer = [[1,2,5,6], []]`.

In [16]:
def trace_dict(matches):
    lost = {}
    for w, l in matches:
        new_w = w not in lost
        lost.setdefault(w, 0)
        lost[l] = lost.get(l, 0) + 1
        print(f"trận ({w} thắng {l}): {'đăng ký ' + str(w) + ' -> 0; ' if new_w else ''}{l} thua -> {lost[l]} | lost = {lost}")
    ans = [sorted(p for p, c in lost.items() if c == 0), sorted(p for p, c in lost.items() if c == 1)]
    print("=>", ans)
    return ans

trace_dict(examples[1][0])

trận (2 thắng 3): đăng ký 2 -> 0; 3 thua -> 1 | lost = {2: 0, 3: 1}
trận (1 thắng 3): đăng ký 1 -> 0; 3 thua -> 2 | lost = {2: 0, 3: 2, 1: 0}
trận (5 thắng 4): đăng ký 5 -> 0; 4 thua -> 1 | lost = {2: 0, 3: 2, 1: 0, 5: 0, 4: 1}
trận (6 thắng 4): đăng ký 6 -> 0; 4 thua -> 2 | lost = {2: 0, 3: 2, 1: 0, 5: 0, 4: 2, 6: 0}
=> [[1, 2, 5, 6], []]


[[1, 2, 5, 6], []]

In [17]:
trace_dict([[1, 2], [2, 3], [3, 1], [4, 1]])

trận (1 thắng 2): đăng ký 1 -> 0; 2 thua -> 1 | lost = {1: 0, 2: 1}
trận (2 thắng 3): 3 thua -> 1 | lost = {1: 0, 2: 1, 3: 1}
trận (3 thắng 1): 1 thua -> 1 | lost = {1: 1, 2: 1, 3: 1}
trận (4 thắng 1): đăng ký 4 -> 0; 1 thua -> 2 | lost = {1: 2, 2: 1, 3: 1, 4: 0}
=> [[4], [2, 3]]


[[4], [2, 3]]

## 9. Code nộp LeetCode

In [18]:
class Solution:
    def findWinners(self, matches: List[List[int]]) -> List[List[int]]:
        lost = {}                                 # người chơi -> số trận thua
        for w, l in matches:
            lost.setdefault(w, 0)                 # người thắng cũng là người đã chơi
            lost[l] = lost.get(l, 0) + 1
        zero = sorted(p for p, c in lost.items() if c == 0)
        one = sorted(p for p, c in lost.items() if c == 1)
        return [zero, one]

for m, exp in examples:
    assert Solution().findWinners(m) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bất biến.** Sau khi xử lý `t` trận đầu, khóa của `lost` đúng là tập người xuất hiện trong `t` trận, và `lost[p]` = số trận `p` thua trong `t` trận đó. Cơ sở `t = 0` hiển nhiên. Bước `t → t+1` với trận `(w, l)`: `setdefault(w, 0)` thêm `w` nếu chưa có mà không đổi số trận thua của `w` (`w` thắng trận này); `lost[l] += 1` (thêm `l` nếu chưa có) cập nhật đúng số trận thua của `l`. Mọi người khác không đổi.

**Đầu ra.** Sau khi xử lý hết, lọc `c == 0` và `c == 1` cho đúng hai tập đề yêu cầu (chỉ gồm người đã chơi); `sorted` cho thứ tự tăng.

**Mảng đếm (Cách 4).** Giá trị lính canh `−1` phân biệt "chưa chơi" với "chơi, 0 trận thua". Duyệt `p` tăng dần cho thứ tự tăng mà không cần sắp.

## 11. Phân tích Time / Space

Gọi `M = len(matches)`, `P` = số người chơi khác nhau (`≤ 2M`), `V` = id lớn nhất (`≤ 10⁵`).

| Cách | Time | Space |
|---|---|---|
| 1. Đếm lại cho từng người | `O(P · M)` | `O(P)` |
| 2. `dict` + sort | `O(M + P log P)` | `O(P)` |
| 3. `Counter` + `set` | `O(M + P log P)` | `O(P)` |
| 4. Mảng đếm | `O(M + V)` | `O(V)` |
| 5. Sort + groupby | `O(M log M)` | `O(M)` |

In [19]:
def gen_matches(M, V, rng):
    seen = set(); out = []
    M = min(M, V * (V - 1))                 # số kết quả trận khác nhau tối đa
    while len(out) < M:
        w, l = rng.randint(1, V), rng.randint(1, V)
        if w != l and (w, l) not in seen:
            seen.add((w, l)); out.append([w, l])
    return out

rng = random.Random(2225)
big = gen_matches(10**5, 10**5, rng)
for name, fn in [("dict", winners_dict), ("counter", winners_counter), ("array", winners_array),
                 ("sort_group", winners_sort_group), ("Solution", lambda m: Solution().findWinners(m))]:
    t = time.perf_counter(); r = fn(big)
    print(f"{name:10s}: |zero|={len(r[0])}, |one|={len(r[1])} ({time.perf_counter() - t:.3f}s)")

dict      : |zero|=23346, |one|=36877 (0.031s)


counter   : |zero|=23346, |one|=36877 (0.046s)
array     : |zero|=23346, |one|=36877 (0.017s)
sort_group: |zero|=23346, |one|=36877 (0.041s)
Solution  : |zero|=23346, |one|=36877 (0.030s)


Kiểm chứng **bất biến sau từng trận**: sau khi xử lý `t` trận đầu, bảng `lost` của Cách 2 phải bằng đúng bảng tính lại từ đầu trên `t` trận đó (cả tập khóa lẫn giá trị).

In [20]:
rng = random.Random(10)
for _ in range(100):
    m = gen_matches(rng.randint(1, 25), rng.randint(2, 8), rng)
    lost = {}
    for t, (w, l) in enumerate(m, 1):
        lost.setdefault(w, 0); lost[l] = lost.get(l, 0) + 1
        prefix = m[:t]
        expect = {p: sum(1 for _, x in prefix if x == p) for match in prefix for p in match}
        assert lost == expect
print("Bất biến của bảng lost đúng sau từng trận (100 chuỗi trận ngẫu nhiên)")

Bất biến của bảng lost đúng sau từng trận (100 chuỗi trận ngẫu nhiên)


## 12. Bộ test

- Ví dụ đề.
- Biên: một trận; vòng tròn `1→2→3→1` (mỗi người thua 1); một người thắng tất cả; người có id lớn nhất `10⁵`.
- Đối chiếu mọi cách với brute force trên dữ liệu ngẫu nhiên (không có hai trận cùng kết quả; seed cố định), và kiểm danh sách **tăng dần**.

In [21]:
fns = {"brute": winners_brute, "dict": winners_dict, "counter": winners_counter, "array": winners_array,
       "sort_group": winners_sort_group, "Solution": lambda m: Solution().findWinners(m),
       "SolutionArray": lambda m: SolutionArray().findWinners(m)}
edge = [([[1, 2]], [[1], [2]]), ([[1, 2], [2, 3], [3, 1]], [[], [1, 2, 3]]),
        ([[1, 2], [1, 3], [1, 4]], [[1], [2, 3, 4]]), ([[100000, 1], [2, 100000]], [[2], [1, 100000]]),
        ([[1, 2], [3, 2], [2, 4]], [[1, 3], [4]])]
for m, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(m) == exp, (name, m, fn(m))
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [22]:
rng = random.Random(42)
for _ in range(300):
    m = gen_matches(rng.randint(1, 40), rng.randint(2, 12), rng)
    want = winners_brute(m)
    for name, fn in fns.items():
        got = fn(m)
        assert got == want and got[0] == sorted(got[0]) and got[1] == sorted(got[1]), (name, m)
print("300 test ngẫu nhiên: mọi cách khớp brute force, đầu ra tăng dần")

300 test ngẫu nhiên: mọi cách khớp brute force, đầu ra tăng dần


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên người **chỉ thắng** (không bao giờ thua)

In [23]:
def wrong_only_losers(matches):
    losses = Counter(l for _, l in matches)
    return [sorted(p for p, c in losses.items() if c == 0),           # SAI: luôn rỗng
            sorted(p for p, c in losses.items() if c == 1)]

m, exp = examples[0]
print("sai:", wrong_only_losers(m), "| đúng:", exp)

sai: [[], [4, 5, 7, 8]] | đúng: [[1, 2, 10], [4, 5, 7, 8]]


In [24]:
print("sửa:", winners_counter(m))

sửa: [[1, 2, 10], [4, 5, 7, 8]]


### Lỗi 2 — Mảng đếm khởi tạo 0 (lẫn người chưa chơi với người không thua)

In [25]:
def wrong_array_zero_init(matches):
    top = max(max(w, l) for w, l in matches)
    lost = [0] * (top + 1)                                             # SAI: không phân biệt "chưa chơi"
    for w, l in matches: lost[l] += 1
    return [[p for p in range(1, top + 1) if lost[p] == 0], [p for p in range(1, top + 1) if lost[p] == 1]]

m = [[5, 7]]
print("sai:", wrong_array_zero_init(m), "| đúng:", Solution().findWinners(m))

sai: [[1, 2, 3, 4, 5, 6], [7]] | đúng: [[5], [7]]


In [26]:
print("sửa (lính canh -1):", SolutionArray().findWinners(m))

sửa (lính canh -1): [[5], [7]]


### Lỗi 3 — Quên sắp xếp đầu ra

Thứ tự duyệt `dict` là thứ tự **chèn**, không phải tăng dần.

In [27]:
def wrong_no_sort(matches):
    lost = {}
    for w, l in matches:
        lost.setdefault(w, 0); lost[l] = lost.get(l, 0) + 1
    return [[p for p, c in lost.items() if c == 0], [p for p, c in lost.items() if c == 1]]   # SAI: chưa sort

m, exp = examples[0]
print("sai:", wrong_no_sort(m), "| đúng:", exp)

sai: [[1, 2, 10], [5, 7, 4, 8]] | đúng: [[1, 2, 10], [4, 5, 7, 8]]


In [28]:
print("sửa:", Solution().findWinners(m))

sửa: [[1, 2, 10], [4, 5, 7, 8]]


### Lỗi 4 — Ghi đè số trận thua khi đăng ký người thắng (`lost[w] = 0`)

Nếu người `w` đã thua trước đó, `lost[w] = 0` xóa mất thông tin. Phải dùng `setdefault` (hoặc `if w not in lost`).

In [29]:
def wrong_overwrite(matches):
    lost = {}
    for w, l in matches:
        lost[w] = 0                                                    # SAI: ghi đè
        lost[l] = lost.get(l, 0) + 1
    return [sorted(p for p, c in lost.items() if c == 0), sorted(p for p, c in lost.items() if c == 1)]

m, exp = examples[0]
print("sai:", wrong_overwrite(m), "| đúng:", exp)

sai: [[1, 2, 3, 10], [4, 5, 7, 8]] | đúng: [[1, 2, 10], [4, 5, 7, 8]]


In [30]:
print("sửa (setdefault):", winners_dict(m))

sửa (setdefault): [[1, 2, 10], [4, 5, 7, 8]]


### Lỗi 5 — Đếm **tổng số trận** (thắng + thua) thay vì số trận thua

In [31]:
def wrong_total_games(matches):
    games = Counter(p for match in matches for p in match)            # SAI: đếm cả trận thắng
    losses = Counter(l for _, l in matches)
    return [sorted(p for p in games if losses[p] == 0), sorted(p for p, c in games.items() if c == 1)]

m, exp = examples[0]
print("sai:", wrong_total_games(m), "| đúng:", exp)

sai: [[1, 2, 10], [1, 2, 7, 8]] | đúng: [[1, 2, 10], [4, 5, 7, 8]]


In [32]:
print("sửa:", Solution().findWinners(m))

sửa: [[1, 2, 10], [4, 5, 7, 8]]


### Lỗi 6 — Lọc `c <= 1` cho `answer[1]` (lẫn người 0 trận thua vào danh sách 1 trận thua)

In [33]:
def wrong_le_one(matches):
    lost = {}
    for w, l in matches:
        lost.setdefault(w, 0); lost[l] = lost.get(l, 0) + 1
    return [sorted(p for p, c in lost.items() if c == 0), sorted(p for p, c in lost.items() if c <= 1)]   # SAI

m, exp = examples[0]
print("sai:", wrong_le_one(m), "| đúng:", exp)

sai: [[1, 2, 10], [1, 2, 4, 5, 7, 8, 10]] | đúng: [[1, 2, 10], [4, 5, 7, 8]]


In [34]:
print("sửa (c == 1):", Solution().findWinners(m))

sửa (c == 1): [[1, 2, 10], [4, 5, 7, 8]]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Liệt kê theo số lần xuất hiện"** ⇒ đếm tần suất bằng `dict`/`Counter`; nhớ **đăng ký** các phần tử có tần suất 0.
- id bị chặn nhỏ (`≤ 10⁵`) ⇒ **mảng đếm** + lính canh, thứ tự có sẵn.
- `dict` giữ thứ tự **chèn**, không phải thứ tự **khóa** ⇒ cần `sorted` nếu đề yêu cầu tăng dần.
- `setdefault` để thêm mà không ghi đè.
- Đọc kỹ "chỉ xét người đã chơi ít nhất một trận".

## 15. Biến thể của bài

1. **LeetCode 997 — Find the Town Judge:** người được mọi người tin và không tin ai ⇒ đếm "bậc vào/bậc ra".
2. **Người thua đúng `k` trận** (tổng quát hóa): lọc `c == k`.
3. **Xếp hạng theo (số trận thua tăng, số trận thắng giảm, id tăng)** (bài tập 5).
4. **LeetCode 1636 — Sort Array by Increasing Frequency:** sắp theo tần suất.

Minh họa biến thể 1 (Town Judge, tự dựng, kiểm bằng brute force):

In [35]:
def find_judge(n, trust):
    score = [0] * (n + 1)
    for a, b in trust:
        score[a] -= 1; score[b] += 1
    for p in range(1, n + 1):
        if score[p] == n - 1: return p
    return -1

def judge_brute(n, trust):
    T = set(map(tuple, trust))
    for p in range(1, n + 1):
        if all((p, q) not in T for q in range(1, n + 1)) and all((q, p) in T for q in range(1, n + 1) if q != p):
            return p
    return -1

rng = random.Random(997)
for _ in range(400):
    n = rng.randint(1, 5)
    trust = [[a, b] for a in range(1, n + 1) for b in range(1, n + 1) if a != b and rng.random() < 0.5]
    if rng.random() < 0.3 and n > 1:                 # cài sẵn một thẩm phán
        j = rng.randint(1, n); trust = [t for t in trust if t[0] != j] + [[q, j] for q in range(1, n + 1) if q != j]
        trust = [list(t) for t in {tuple(t) for t in trust}]
    assert find_judge(n, trust) == judge_brute(n, trust)
print("Town Judge khớp brute force; n=3, [[1,3],[2,3]] ->", find_judge(3, [[1, 3], [2, 3]]))

Town Judge khớp brute force; n=3, [[1,3],[2,3]] -> 3


Minh họa biến thể 4 — **LeetCode 1636 (Sort Array by Increasing Frequency)**: sắp mảng theo tần suất **tăng**, cùng tần suất thì giá trị **giảm**. Cùng công cụ: `Counter` + `sorted` với khóa tuple. Kiểm bằng cách so sánh từng cặp kề nhau.

In [36]:
def frequency_sort(nums):
    cnt = Counter(nums)
    return sorted(nums, key=lambda x: (cnt[x], -x))

def frequency_sort_ok(nums, out):
    cnt = Counter(nums)
    if Counter(out) != cnt: return False
    return all((cnt[a], -a) <= (cnt[b], -b) for a, b in zip(out, out[1:]))

rng = random.Random(1636)
for _ in range(300):
    a = [rng.randint(-5, 5) for _ in range(rng.randint(1, 15))]
    assert frequency_sort_ok(a, frequency_sort(a))
print("Sort by frequency hợp lệ trên 300 mảng; [1,1,2,2,2,3] ->", frequency_sort([1, 1, 2, 2, 2, 3]))

Sort by frequency hợp lệ trên 300 mảng; [1,1,2,2,2,3] -> [3, 1, 1, 2, 2, 2]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đếm lại | mỗi người duyệt mọi trận | `O(P·M)` | `O(P)` | kiểm tra |
| 2. `dict` + sort | `setdefault`, đếm thua | `O(M + P log P)` | `O(P)` | **nộp bài** |
| 3. `Counter` + `set` | tách đếm và đăng ký | `O(M + P log P)` | `O(P)` | dễ đọc |
| 4. Mảng đếm | lính canh `−1` | `O(M + V)` | `O(V)` | id nhỏ, không cần sort |
| 5. Sort + groupby | không dùng băm | `O(M log M)` | `O(M)` | khi cấm hash |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `findWinners([[1, 2], [2, 1]])`.
2. **Tính tay:** `findWinners([[3, 1], [2, 1], [4, 2], [4, 3], [5, 4]])`.
3. **Điền TODO:** `loss_table(matches)` trả về `dict` người chơi → số trận thua (gồm cả người 0 trận thua).
4. **Tự viết lại:** viết lại Cách 2 không nhìn code.
5. **Mini challenge:** `ranking(matches)` — danh sách mọi người chơi sắp theo (số trận thua **tăng**, số trận thắng **giảm**, id **tăng**). Kiểm bằng brute force.

In [37]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def loss_table(matches):
    # TODO
    pass

In [38]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_find_winners(matches):
    # TODO
    pass

In [39]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def ranking(matches):
    # TODO: đếm thắng và thua, sorted(players, key=lambda p: (lost[p], -won[p], p))
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Mỗi người thua 1 trận. (2) Không ai 0 trận thua. (3) `[[], [1, 2]]`.

**Bài 2.** (1) Thua: 1 thua 2 trận, 2 thua 1, 3 thua 1, 4 thua 1, 5 thua 0. (2) `answer[0] = [5]`. (3) `answer[1] = [2, 3, 4]`.

**Bài 3.** (1) `lost = {}`. (2) `setdefault(w, 0)`. (3) `lost[l] = lost.get(l, 0) + 1`.

**Bài 4.** (1) Bảng thua như Bài 3. (2) Lọc `== 0`, `== 1`. (3) `sorted`.

**Bài 5.** (1) Hai `Counter`: `won`, `lost`. (2) Tập người chơi. (3) Khóa sắp `(lost[p], −won[p], p)`.

## 19. Đáp án tham khảo

In [40]:
# Bài 1, 2
assert Solution().findWinners([[1, 2], [2, 1]]) == [[], [1, 2]]
assert Solution().findWinners([[3, 1], [2, 1], [4, 2], [4, 3], [5, 4]]) == [[5], [2, 3, 4]]
print("Bài 1: [[], [1, 2]] | Bài 2: [[5], [2, 3, 4]]")
trace_dict([[3, 1], [2, 1], [4, 2], [4, 3], [5, 4]])

Bài 1: [[], [1, 2]] | Bài 2: [[5], [2, 3, 4]]
trận (3 thắng 1): đăng ký 3 -> 0; 1 thua -> 1 | lost = {3: 0, 1: 1}
trận (2 thắng 1): đăng ký 2 -> 0; 1 thua -> 2 | lost = {3: 0, 1: 2, 2: 0}
trận (4 thắng 2): đăng ký 4 -> 0; 2 thua -> 1 | lost = {3: 0, 1: 2, 2: 1, 4: 0}
trận (4 thắng 3): 3 thua -> 1 | lost = {3: 1, 1: 2, 2: 1, 4: 0}
trận (5 thắng 4): đăng ký 5 -> 0; 4 thua -> 1 | lost = {3: 1, 1: 2, 2: 1, 4: 1, 5: 0}
=> [[5], [2, 3, 4]]


[[5], [2, 3, 4]]

In [41]:
# Bài 3, 4
def loss_table_ans(matches):
    lost = {}
    for w, l in matches:
        lost.setdefault(w, 0); lost[l] = lost.get(l, 0) + 1
    return lost

def my_find_winners_ans(matches):
    lost = loss_table_ans(matches)
    return [sorted(p for p, c in lost.items() if c == 0), sorted(p for p, c in lost.items() if c == 1)]

rng = random.Random(3)
for _ in range(200):
    m = gen_matches(rng.randint(1, 30), rng.randint(2, 10), rng)
    t = loss_table_ans(m)
    assert set(t) == {p for x in m for p in x} and all(t[p] == sum(1 for _, l in m if l == p) for p in t)
    assert my_find_winners_ans(m) == winners_brute(m)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [42]:
# Bài 5
def ranking_ans(matches):
    won = Counter(w for w, _ in matches); lost = Counter(l for _, l in matches)
    players = {p for x in matches for p in x}
    return sorted(players, key=lambda p: (lost[p], -won[p], p))

def ranking_brute(matches):
    players = sorted({p for x in matches for p in x})
    stats = [(sum(1 for _, l in matches if l == p), -sum(1 for w, _ in matches if w == p), p) for p in players]
    return [p for _, _, p in sorted(stats)]

rng = random.Random(5)
for _ in range(200):
    m = gen_matches(rng.randint(1, 30), rng.randint(2, 10), rng)
    assert ranking_ans(m) == ranking_brute(m)
print("Bài 5: OK; ví dụ 1 ->", ranking_ans(examples[0][0]))

Bài 5: OK; ví dụ 1 -> [10, 1, 2, 4, 5, 7, 8, 3, 6, 9]


**Giải thích thêm cho Bài 5.** Khóa sắp là tuple `(lost[p], −won[p], p)`: Python so tuple theo từng thành phần, nên ai thua ít hơn đứng trước; hòa số trận thua thì ai thắng **nhiều** hơn đứng trước (đổi dấu `−won` để "nhiều" thành "nhỏ"); hòa tiếp thì id nhỏ đứng trước. Bảng dưới in xếp hạng của ví dụ 1 kèm thống kê.

In [43]:
m = examples[0][0]
won = Counter(w for w, _ in m); lost_c = Counter(l for _, l in m)
for rank, p in enumerate(ranking_ans(m), 1):
    print(f"hạng {rank:2d}: người {p:2d} — thua {lost_c[p]}, thắng {won[p]}, khóa = {(lost_c[p], -won[p], p)}")

hạng  1: người 10 — thua 0, thắng 2, khóa = (0, -2, 10)
hạng  2: người  1 — thua 0, thắng 1, khóa = (0, -1, 1)
hạng  3: người  2 — thua 0, thắng 1, khóa = (0, -1, 2)
hạng  4: người  4 — thua 1, thắng 3, khóa = (1, -3, 4)
hạng  5: người  5 — thua 1, thắng 2, khóa = (1, -2, 5)
hạng  6: người  7 — thua 1, thắng 0, khóa = (1, 0, 7)
hạng  7: người  8 — thua 1, thắng 0, khóa = (1, 0, 8)
hạng  8: người  3 — thua 2, thắng 1, khóa = (2, -1, 3)
hạng  9: người  6 — thua 2, thắng 0, khóa = (2, 0, 6)
hạng 10: người  9 — thua 2, thắng 0, khóa = (2, 0, 9)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
lost = {}
for w, l in matches:
    lost.setdefault(w, 0)            # đăng ký người thắng
    lost[l] = lost.get(l, 0) + 1     # đếm trận thua
return [sorted(p for p, c in lost.items() if c == 0),
        sorted(p for p, c in lost.items() if c == 1)]
```

In [44]:
def template(matches):
    lost = {}
    for w, l in matches:
        lost.setdefault(w, 0); lost[l] = lost.get(l, 0) + 1
    return [sorted(p for p, c in lost.items() if c == 0), sorted(p for p, c in lost.items() if c == 1)]
print([template(m) for m, _ in examples])

[[[1, 2, 10], [4, 5, 7, 8]], [[1, 2, 5, 6], []]]


**Checklist:**

- [ ] Người **chỉ thắng** cũng phải có trong bảng (0 trận thua).
- [ ] `setdefault` / kiểm tra trước khi đăng ký — không ghi đè số trận thua.
- [ ] Đếm **trận thua**, không đếm tổng số trận.
- [ ] Hai danh sách **tăng dần** (`sorted`).
- [ ] Mảng đếm: lính canh `−1` cho người chưa chơi.
- [ ] Đã thử: một trận, vòng tròn, một người thắng tất cả.